In [4]:
import requests
import folium
from geopy.geocoders import Nominatim
from datetime import datetime
import pandas as pd
from scipy.spatial import KDTree
from geopy.distance import geodesic
import os
import webbrowser

def get_project_root():
    """Get the project root directory by going up one level from the current directory"""
    current_dir = os.getcwd()
    return os.path.dirname(current_dir)

def load_gtfs_data(project_root):
    """Load all required GTFS files"""
    data_dir = os.path.join(project_root, 'mpt_data', '3')
    
    stops_df = pd.read_csv(os.path.join(data_dir, 'stops.txt'))
    routes_df = pd.read_csv(os.path.join(data_dir, 'routes.txt'))
    trips_df = pd.read_csv(os.path.join(data_dir, 'trips.txt'))
    stop_times_df = pd.read_csv(os.path.join(data_dir, 'stop_times.txt'))
    shapes_df = pd.read_csv(os.path.join(data_dir, 'shapes.txt')) if os.path.exists(os.path.join(data_dir, 'shapes.txt')) else None
    
    print(f"Loaded {len(stops_df)} stops")
    print(f"Loaded {len(routes_df)} routes")
    print(f"Loaded {len(trips_df)} trips")
    print(f"Loaded {len(stop_times_df)} stop times")
    
    return stops_df, routes_df, trips_df, stop_times_df, shapes_df

def geocode_address(address):
    geolocator = Nominatim(user_agent="mapping_app1.0")
    
    #Define the bounding box for Melbourne
    melbourne_bbox = [(-38.5267, 144.5937), (-37.5113, 145.5125)] 
    
    #Geocode the address within the Melbourne bounding box
    location = geolocator.geocode(address, viewbox=melbourne_bbox, bounded=True)
    
    if location:
        return location.latitude, location.longitude
    else:
        print("Address not found within Melbourne.")
        return None

def find_nearest_stop(lat, lon, kdtree, df):
    distance, index = kdtree.query([lat, lon])
    nearest_stop = df.iloc[index]
    
    stop_coords = (nearest_stop["stop_lat"], nearest_stop["stop_lon"])
    point_coords = (lat, lon)
    distance_meters = geodesic(point_coords, stop_coords).meters
    
    print(f"\nNearest stop: {nearest_stop['stop_name']}")
    print(f"Stop ID: {nearest_stop['stop_id']}")
    print(f"Distance: {distance_meters:.0f} meters")
    
    return nearest_stop, distance_meters, distance

def find_nearest_tram_stop(lat, lon, tram_kdtree, tram_df):
    distance, index = tram_kdtree.query([lat, lon])
    nearest_tram_stop = tram_df.iloc[index]
    
    nearest_tram_stop_coords = (nearest_tram_stop["stop_lat"], nearest_tram_stop["stop_lon"])
    point_coords = (lat, lon)
    
    distance_meters = geodesic(point_coords, nearest_tram_stop_coords).meters
    return nearest_tram_stop, distance_meters

def main():
    print("\nBus Route Planner")
    print("-----------------")
    project_route = get_project_root()
    print(project_route)
    stops_df, routes_df, trips_df, stop_times_df, shapes_df = load_gtfs_data(project_route)
    coords = stops_df[["stop_lat", "stop_lon"]].values
    kdtree = KDTree(coords)

    #Get user input
    current_location = input("\nEnter your starting location: ")
    destination_input = input("Enter your destination: ")

    start_coords = geocode_address(current_location)
    if not start_coords:
        raise ValueError("could not find starting location")

    print("found you bitch")

    nearest_start_stop, start_distance, _ = find_nearest_stop(start_coords[0], start_coords[1], kdtree, stops_df)

    print(nearest_start_stop)

if __name__ == "__main__":
    main()


Bus Route Planner
-----------------
C:\Users\jubal\Documents\SIT374\MOP\MOP-Code\artificial-intelligence\chatbot\code\mpt_bot
Loaded 1627 stops
Loaded 48 routes
Loaded 35414 trips
Loaded 1671407 stop times



Enter your starting location:  flagstaff
Enter your destination:  parliament


found you bitch

Nearest stop: 7-Flagstaff Railway Station/William St (Melbourne City)
Stop ID: 18201
Distance: 46 meters
stop_id                                                  18201
stop_name    7-Flagstaff Railway Station/William St (Melbou...
stop_lat                                            -37.811996
stop_lon                                            144.956517
Name: 290, dtype: object
